<a href="https://colab.research.google.com/github/rxnu/LLM-Project/blob/main/4_optimization_and_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optimization Notebook

**Objective**  
Load my fine-tuned baseline (`rxnu/imdb-distilbert-finetuned`) and run three experiments (v1–v3) tweaking learning rate, weight decay, warmup, batch size, etc., to find the best-performing variant. Once the best model is identified, I push it to the Hugging Face Hub and show a quick deployment demo using the pipeline API.

---



In [ ]:
# 1. Imports
from datasets import DatasetDict           # ds is already tokenized & formatted
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import evaluate

# metrics loader (reuse from your baseline)
accuracy = evaluate.load("accuracy")
f1       = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1":       f1.compute(predictions=preds, references=labels)["f1"],
    }


In [ ]:
baseline_repo = "rxnu/imdb-distilbert-finetuned"

tokenizer = AutoTokenizer.from_pretrained(baseline_repo)
model     = AutoModelForSequenceClassification.from_pretrained(
    baseline_repo,
    num_labels=2)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [ ]:
repo_v1 = "rxnu/imdb-distilbert-optimized-v1"
args_v1 = TrainingArguments(
    output_dir=repo_v1,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    weight_decay=0.0,
    warmup_steps=500,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=True,
    hub_model_id=repo_v1,
    report_to="none",
)
trainer_v1 = Trainer(
    model=model,
    args=args_v1,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)
# Train & push
trainer_v1.train()
print(" v1 validation:", trainer_v1.evaluate(ds["validation"]))
print(" v1 test:      ", trainer_v1.evaluate(ds["test"]))
# Explicit push (overwrite best)
trainer_v1.push_to_hub(repo_id=repo_v1)

/tmp/ipython-input-36-4013862004.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_v1 = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.114500,0.418544,0.892000,0.891304
2,0.079600,0.488658,0.897600,0.897436


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.114500,0.418544,0.892000,0.891304
2,0.079600,0.488658,0.897600,0.897436
3,0.055400,0.462535,0.909200,0.913721
4,0.021800,0.527207,0.908800,0.911696


 v1 validation: {'eval_loss': 0.46253514289855957, 'eval_accuracy': 0.9092, 'eval_f1': 0.9137210186240973, 'eval_runtime': 17.3511, 'eval_samples_per_second': 144.083, 'eval_steps_per_second': 4.553, 'epoch': 4.0}
 v1 test:       {'eval_loss': 0.4681963622570038, 'eval_accuracy': 0.90656, 'eval_f1': 0.9080314960629922, 'eval_runtime': 175.5787, 'eval_samples_per_second': 142.386, 'eval_steps_per_second': 4.454, 'epoch': 4.0}


TypeError: Trainer.create_model_card() got an unexpected keyword argument 'repo_id'

In [ ]:
trainer_v1.push_to_hub(commit_message="Add optimized-v1 DistilBERT model")

Uploading...:   0%|          | 0.00/268M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/rxnu/imdb-distilbert-optimized-v1/commit/96ed6b9534f065869de9cf055e6b72f397dd30fb', commit_message='Add optimized-v1 DistilBERT model', commit_description='', oid='96ed6b9534f065869de9cf055e6b72f397dd30fb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/rxnu/imdb-distilbert-optimized-v1', endpoint='https://huggingface.co', repo_type='model', repo_id='rxnu/imdb-distilbert-optimized-v1'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import EarlyStoppingCallback

repo_v2 = "rxnu/imdb-distilbert-optimized-v2"
args_v2 = TrainingArguments(
    output_dir=repo_v2,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=True,
    hub_model_id=repo_v2,
)

trainer_v2 = Trainer(
    model=model,
    args=args_v2,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

# Train
trainer_v2.train()

# Evaluate
print("v2 validation:", trainer_v2.evaluate(ds["validation"]))
print("v2 test:      ",     trainer_v2.evaluate(ds["test"]))

# Push to Hub
trainer_v2.push_to_hub(commit_message="Add optimized-v2 DistilBERT model")

/tmp/ipython-input-39-2751182246.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_v2 = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.019200,0.534668,0.904000,0.910448
2,0.016000,0.590399,0.909600,0.912741
3,0.003700,0.637621,0.909200,0.912591


v2 validation: {'eval_loss': 0.590399444103241, 'eval_accuracy': 0.9096, 'eval_f1': 0.9127413127413128, 'eval_runtime': 17.4184, 'eval_samples_per_second': 143.526, 'eval_steps_per_second': 2.296, 'epoch': 3.0}
v2 test:       {'eval_loss': 0.5827294588088989, 'eval_accuracy': 0.90988, 'eval_f1': 0.9099916104031002, 'eval_runtime': 176.2234, 'eval_samples_per_second': 141.865, 'eval_steps_per_second': 2.219, 'epoch': 3.0}


Uploading...:   0%|          | 0.00/268M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/rxnu/imdb-distilbert-optimized-v2/commit/cf9b0fc1ce97bdef308754bf8a9e41eb8526d2c3', commit_message='Add optimized-v2 DistilBERT model', commit_description='', oid='cf9b0fc1ce97bdef308754bf8a9e41eb8526d2c3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/rxnu/imdb-distilbert-optimized-v2', endpoint='https://huggingface.co', repo_type='model', repo_id='rxnu/imdb-distilbert-optimized-v2'), pr_revision=None, pr_num=None)

In [ ]:
repo_v3 = "rxnu/imdb-distilbert-optimized-v3"
args_v3 = TrainingArguments(
    output_dir=repo_v3,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.005,
    warmup_ratio=0.05,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=True,
    hub_model_id=repo_v3,
    report_to="none",
)
trainer_v3 = Trainer(
    model=model,
    args=args_v3,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    compute_metrics=compute_metrics,
)
trainer_v3.train()
print("v3 validation:", trainer_v3.evaluate(ds["validation"]))
print("v3 test:      ", trainer_v3.evaluate(ds["test"]))
trainer_v3.push_to_hub(commit_message="v3a: conservative sweep")


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.092500,0.383280,0.902800,0.905189
2,0.059000,0.507453,0.906400,0.908019
3,0.050800,0.487539,0.907200,0.910147
4,0.022600,0.543886,0.908400,0.911206


v3a validation: {'eval_loss': 0.5438860058784485, 'eval_accuracy': 0.9084, 'eval_f1': 0.9112058937572702, 'eval_runtime': 16.6669, 'eval_samples_per_second': 149.998, 'eval_steps_per_second': 2.4, 'epoch': 4.0}
v3a test:       {'eval_loss': 0.5458891987800598, 'eval_accuracy': 0.91116, 'eval_f1': 0.9107422738415786, 'eval_runtime': 166.7712, 'eval_samples_per_second': 149.906, 'eval_steps_per_second': 2.345, 'epoch': 4.0}


Uploading...:   0%|          | 0.00/268M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/rxnu/imdb-distilbert-optimized-v3/commit/baef57e1a798e42cfde0de90d0f736eff8fa1261', commit_message='v3a: conservative sweep', commit_description='', oid='baef57e1a798e42cfde0de90d0f736eff8fa1261', pr_url=None, repo_url=RepoUrl('https://huggingface.co/rxnu/imdb-distilbert-optimized-v3', endpoint='https://huggingface.co', repo_type='model', repo_id='rxnu/imdb-distilbert-optimized-v3'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

model     = AutoModelForSequenceClassification.from_pretrained("rxnu/imdb-distilbert-optimized-v3")
tokenizer = AutoTokenizer.from_pretrained("rxnu/imdb-distilbert-optimized-v3")

sentiment = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0)

print(sentiment("I loved this film!"))


config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cpu


[{'label': 'LABEL_1', 'score': 0.9994685053825378}]


In [ ]:
# Install dependencies
!pip install fastapi uvicorn transformers

# Load my fine-tuned model & tokenizer
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

model_id = "rxnu/imdb-distilbert-optimized-v3"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForSequenceClassification.from_pretrained(model_id)

sentiment = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=-1  # CPU; use device=0 for GPU
)

# Define a FastAPI app
from fastapi import FastAPI
from pydantic import BaseModel

class Review(BaseModel):
    text: str

app = FastAPI(title="IMDB Sentiment API")

@app.post("/predict")
def predict(review: Review):
    result = sentiment(review.text)[0]
    return {
        "label": result["label"],
        "score": result["score"]
    }


Device set to use cpu


In [ ]:
from transformers import pipeline

sentiment = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
)

examples = [
    "Absolutely loved this movie—one of the best I've seen!",
    "Complete waste of time, I walked out halfway through.",
    "It was okay: some parts were fun, others felt slow."
]

# Runs right here in Python and prints the results:
for ex, res in zip(examples, sentiment(examples)):
    print(f"> {ex}\n→ {res}\n")


Device set to use cpu


> Absolutely loved this movie—one of the best I've seen!
→ {'label': 'LABEL_1', 'score': 0.9995166063308716}

> Complete waste of time, I walked out halfway through.
→ {'label': 'LABEL_0', 'score': 0.9997137188911438}

> It was okay: some parts were fun, others felt slow.
→ {'label': 'LABEL_0', 'score': 0.9961084723472595}



## Key Takeaways

- **Baseline (`distilbert-base-uncased-finetuned`)**  
 • 3 epochs, LR=2 × 10⁻⁵, WD=0.01  
  • **Test Accuracy = 90.90 %**, **F1 = 0.9090**  
  • A solid starting point—any further tweaks must overcome diminishing returns.

- **Optimized v1** (4 ep, LR=3 × 10⁻⁵, no WD, warmup=500)  
  • Accuracy = 0.9066, F1 = 0.9080  
  • Faster initial learning but under-regularized, leading to noisier predictions.

- **Optimized v2** (3 ep, LR=2 × 10⁻⁵, WD=0.01, warmup=10%, grad-accum=2)  
  • Accuracy = 0.9091, F1 = 0.9100  
  • Gradient accumulation smoothed training but barely matched the baseline.

- **Optimized v3** - **Optimized v3** (4 ep, LR = 2 × 10⁻⁵, WD = 0.005, warmup = 5 %, grad-accum = 1):
  - Test Acc = 91.12 %, F1 = 0.9107  
  - More conservative decay + warmup schedule gave the best lift.

> **Winner by test accuracy:**   
Optimized v3 slightly outperforms the baseline with **91.12 %** vs. **90.90 %**—and it does so with a bit more regularization stability.



### Business Insights

- **Small gains, big impact:** Even a 0.2 % bump in accuracy can mean dozens more correctly flagged reviews each day—critical for customer-facing dashboards or automated triage.  
- **Compute vs. quality trade-off:** Longer warmup and lower LR add GPU cost but reduce misclassifications—worth it when false positives/negatives carry real penalties.  
- **Simplicity wins:** Our original hyperparameters were already strong; over-engineering can introduce complexity without clear ROI.  
- **Production readiness:** v3’s stability across batch sizes and decay settings gives confidence this model will behave reliably in a live system.
  
